In [ ]:
# Data handling
import numpy as np
import pandas as pd
import os

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Image processing
from PIL import Image

# Deep Learning
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.optimizers import Adam

# Evaluation
from sklearn.metrics import classification_report, confusion_matrix

# Utility
import random

import shutil
import random




In [ ]:
base_path = "data/garbage_classification"

for category in os.listdir(base_path):
    print(category, ":", len(os.listdir(os.path.join(base_path, category))))

In [ ]:
import os
import shutil
import random

source_dir = "data/garbage_classification"
train_dir = "data/train"
test_dir = "data/test"

if os.path.exists(train_dir):
    shutil.rmtree(train_dir)

if os.path.exists(test_dir):
    shutil.rmtree(test_dir)

os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

for class_name in os.listdir(source_dir):
    class_path = os.path.join(source_dir, class_name)

    if not os.path.isdir(class_path):
        continue

    files = os.listdir(class_path)
    random.shuffle(files)

    split = int(0.8 * len(files))

    train_files = files[:split]
    test_files = files[split:]

    os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, class_name), exist_ok=True)

    for f in train_files:
        shutil.copy(
            os.path.join(class_path, f),
            os.path.join(train_dir, class_name, f)
        )

    for f in test_files:
        shutil.copy(
            os.path.join(class_path, f),
            os.path.join(test_dir, class_name, f)
        )

print("Data split completed successfully!")

In [ ]:
print("\nTRAIN DATA:")
for category in os.listdir(train_dir):
    count = len(os.listdir(os.path.join(train_dir, category)))
    print(category, ":", count)

print("\nTEST DATA:")
for category in os.listdir(test_dir):
    count = len(os.listdir(os.path.join(test_dir, category)))
    print(category, ":", count)

In [ ]:
for category in os.listdir(source_dir):
    print(category, ":", len(os.listdir(os.path.join(source_dir, category))))
    
    train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.3,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    shear_range=0.2
)

    val_datagen = ImageDataGenerator(rescale=1./255)
    
    train_generator = train_datagen.flow_from_directory(
    "data/train",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)
    
    val_generator = val_datagen.flow_from_directory(
    "data/test",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False   # 🔥 IMPORTANT
)

print(train_generator.class_indices)

In [ ]:
y_train = train_generator.classes

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

In [ ]:
images, labels = next(train_generator)

plt.figure(figsize=(10,10))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(images[i])
    plt.axis('off')

plt.show()

In [ ]:
categories = []
counts = []

for category in os.listdir(source_dir):
    path = os.path.join(source_dir, category)
    categories.append(category)
    counts.append(len(os.listdir(path)))

print(list(zip(categories, counts)))

plt.figure(figsize=(8,5))
plt.bar(categories, counts)
plt.xticks(rotation=45)
plt.title("Number of Images per Class")
plt.xlabel("Classes")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(10,10))

for i, category in enumerate(categories):
    folder = os.path.join(source_dir, category)
    img_name = random.choice(os.listdir(folder))
    img_path = os.path.join(folder, img_name)

    img = Image.open(img_path)

    plt.subplot(3,3,i+1)
    plt.imshow(img)
    plt.title(category)
    plt.axis('off')

plt.show()

In [ ]:
sample_images = []

for category in categories:
    folder = os.path.join(source_dir, category)
    img_name = random.choice(os.listdir(folder))
    img_path = os.path.join(folder, img_name)

    img = Image.open(img_path).resize((224,224))
    img_array = np.array(img)

    sample_images.append(img_array)

sample_images = np.array(sample_images)

plt.figure(figsize=(8,5))

plt.hist(sample_images.flatten(), bins=50)
plt.title("Pixel Intensity Distribution")
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# MobileNetV2 Model

base_model_m = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model_m.trainable = True
for layer in base_model_m.layers[:-20]:
    layer.trainable = False

model_m = models.Sequential([
    base_model_m,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(6, activation='softmax')   # 6 classes
])

model_m.compile(
    optimizer=Adam(learning_rate=0.0001),   # 🔥 IMPORTANT
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_m = model_m.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    class_weight=class_weights
)

plt.plot(history_m.history['accuracy'], label='Train Accuracy')
plt.plot(history_m.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title("Accuracy")
plt.show()

plt.plot(history_m.history['loss'], label='Train Loss')
plt.plot(history_m.history['val_loss'], label='Val Loss')
plt.legend()
plt.title("Loss")
plt.show()  

In [ ]:
val_generator.reset()

y_pred_m = model_m.predict(val_generator)   # or model_m
y_pred_classes = np.argmax(y_pred_m, axis=1)

y_true = val_generator.classes
class_labels = list(val_generator.class_indices.keys())

In [ ]:
cm = confusion_matrix(y_true, y_pred_classes)
print(cm)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
filepaths = val_generator.filepaths

errors = np.where(y_pred_classes != y_true)[0]

plt.figure(figsize=(10,10))

for i, idx in enumerate(errors[:9]):
    img_path = filepaths[idx]

    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img) / 255.0

    plt.subplot(3,3,i+1)
    plt.imshow(img_array)
    plt.title(f"True: {class_labels[y_true[idx]]}\nPred: {class_labels[y_pred_classes[idx]]}")
    plt.axis('off')

plt.show()

loss, accuracy = model_m.evaluate(val_generator)
print("Validation Accuracy:", accuracy)

In [ ]:
# ResNet50 Model

base_model_r = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model_r.trainable = True
for layer in base_model_r.layers[:-20]:
    layer.trainable = False

model_r = models.Sequential([
    base_model_r,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(6, activation='softmax')
])

model_r.compile(
    optimizer=Adam(learning_rate=0.0001),   # 🔥 IMPORTANT
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_r = model_r.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    class_weight=class_weights
)

In [ ]:
val_generator.reset()

y_pred_r = model_r.predict(val_generator)   # or model_m
y_pred_classes = np.argmax(y_pred_r, axis=1)

y_true = val_generator.classes

print(classification_report(y_true, y_pred_classes))

In [ ]:
val_generator.reset()

y_pred_m = model_m.predict(val_generator)
y_pred_m = np.argmax(y_pred_m, axis=1)


y_pred_r = model_r.predict(val_generator)
y_pred_r = np.argmax(y_pred_r, axis=1)

y_true = val_generator.classes
class_labels = list(val_generator.class_indices.keys())

report_m = classification_report(y_true, y_pred_m, target_names=class_labels, output_dict=True)

report_r = classification_report(y_true, y_pred_r, target_names=class_labels, output_dict=True)

f1_m = report_m['weighted avg']['f1-score']
f1_r = report_r['weighted avg']['f1-score']

In [ ]:

loss_r, acc_r = model_r.evaluate(val_generator)

results = pd.DataFrame({
    "Model": ["MobileNetV2", "ResNet50"],
    "Accuracy": [accuracy, acc_r],
    "F1-score": [f1_m, f1_r]
})

print(results)

best_model = results.loc[results['F1-score'].idxmax()]

print("Best Model:")
print(best_model)

model_m.save("models/best_model.h5")